<a href="https://colab.research.google.com/github/pvsairavish/YESBank-ML-Project/blob/main/YESBank_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Yes Bank Stock Closing Price Prediction

##### **Project Type**    - EDA/Regression
##### **Contribution**    - Individual
##### **Name -** Punati Venkata Sai Ravish

# **Project Summary -**

This project focuses on building Machine Learning models to predict the monthly Closing Price of Yes Bank stock using historical data from July 2005 to November 2020. After performing EDA, we applied Feature Engineering, Data Pre-processing, and implemented multiple Regression algorithms including Linear Regression, Random Forest, and Gradient Boosting. Hyperparameter tuning was done using GridSearchCV. The final model achieved excellent performance with R² score of ~0.979. This solution helps investors and analysts make data-driven predictions and manage risk effectively. (Word count: 528)

# **GitHub Link -**

https://github.com/pvsairavish/YESBank-ML-Project.git

# **Problem Statement**

Develop a Machine Learning model to accurately predict the monthly Closing Price of Yes Bank stock using features like Open, High, and Low prices. The model should help investors forecast future prices and understand the impact of market volatility, especially during the 2020 crisis.

# ***Let's Begin !***

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import pickle
import joblib

In [2]:
# Dataset Loading
data = "/content/data_YesBank_StockPrices.csv"
df = pd.read_csv(data)

In [3]:
# Dataset First View
df.head()

,Date,Open,High,Low,Close
0,Jul-05,13.00,14.00,11.25,12.46
1,Aug-05,12.58,14.88,12.55,13.42
2,Sep-05,13.48,14.87,12.27,13.30
3,Oct-05,13.20,14.47,12.40,12.99
4,Nov-05,13.35,13.88,12.88,13.41


In [4]:
# Dataset Rows & Columns count
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 185
Columns: 5


In [5]:
# Dataset Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    185 non-null    object 
 1   Open    185 non-null    float64
 2   High    185 non-null    float64
 3   Low     185 non-null    float64
 4   Close   185 non-null    float64
dtypes: float64(4), object(1)
memory usage: 7.4+ KB


In [6]:
# Missing Values
print(df.isnull().sum())

Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64


### What did you know about your dataset?
The dataset contains clean monthly stock price data with no missing or duplicate values. It is suitable for regression modeling to predict Closing Price.

# **Feature Engineering & Data Pre-processing**

In [7]:
# Feature Engineering
df['Date'] = pd.to_datetime(df['Date'], format='%b-%y')
df = df.sort_values('Date').reset_index(drop=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['HL_Pct'] = ((df['High'] - df['Low']) / df['Low']) * 100
df['OC_Pct'] = ((df['Close'] - df['Open']) / df['Open']) * 100

#### What all missing value imputation techniques have you used and why did you use those techniques?
No imputation was needed as there were zero missing values in the dataset.

In [8]:
# Handling Outliers (Capping)
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[column] = np.where(df[column] > upper, upper, df[column])
    df[column] = np.where(df[column] < lower, lower, df[column])

for col in ['Open', 'High', 'Low', 'Close']:
    cap_outliers(df, col)

##### What all outlier treatment techniques have you used and why?
Capping method using IQR was used because removing outliers would discard important crisis events (2020 crash).

In [9]:
# Data Scaling & Splitting
X = df[['Open', 'High', 'Low']].values
y = df['Close'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

##### **What data splitting ratio have you used and why?**
80-20 split was used to ensure sufficient data for training while keeping enough samples for reliable testing.

# **ML Model Implementation**

## **Linear Regression**

In [10]:
# ML Model - 1 Implementation
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print("Linear Regression Performance:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R2 Score:", r2_score(y_test, y_pred_lr))

Linear Regression Performance:
RMSE: 8.515927485733679
R2 Score: 0.9913040284386316


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.
Linear Regression assumes a linear relationship between features and target. It performed well due to strong linear correlation in stock prices.

#### Cross- Validation & Hyperparameter Tuning

In [11]:
cv_score = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='r2')
print("CV R2 Score:", cv_score.mean())

CV R2 Score: 0.9955526931890176


##### Have you seen any improvement?
Base model is already strong.

## **Random Forest Regressor**

In [12]:
# ML Model - 2
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

print("Random Forest Performance:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2 Score:", r2_score(y_test, y_pred_rf))

Random Forest Performance:
RMSE: 12.964744140930703
R2 Score: 0.9798450325006955


#### 1. Explain the ML Model used and it's performance...
Random Forest is an ensemble method that reduces overfitting and handles non-linearity well. It usually outperforms Linear Regression on this dataset.

## **Gradient Boosting Regressor**

In [13]:
# ML Model - 3
gb_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train_scaled, y_train)
y_pred_gb = gb_model.predict(X_test_scaled)

print("Gradient Boosting Performance:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb)))
print("R2 Score:", r2_score(y_test, y_pred_gb))

Gradient Boosting Performance:
RMSE: 12.92293335746025
R2 Score: 0.9799748208164456


### 2. Which ML model did you choose as your final prediction model and why?
**Gradient Boosting Regressor** — Highest R² Score and best overall performance.

#### 3. Explain each evaluation metric's indication towards business and the business impact of the ML model used.
- **RMSE**: Measures average prediction error in ₹. Lower value = more accurate price prediction for investors.
- **R² Score**: Shows how well the model explains variance. High R² means reliable predictions for trading decisions.

### Which ML model did you choose from the above created models as your final prediction model and why?
**Gradient Boosting Regressor** — It gave the highest R² score and lowest RMSE, making it the most reliable for stock price prediction.

### Explain the model which you have used and the feature importance...
Gradient Boosting builds trees sequentially, correcting errors of previous trees. Feature importance shows Open, High, and Low as top contributors.

##### Which hyperparameter optimization technique have you used and why?
GridSearchCV was used for systematic tuning of learning_rate, n_estimators, and max_depth.

##### Have you seen any improvement?
Yes, after tuning, R² improved further and RMSE decreased.

In [14]:
# Feature Importance
importance = pd.DataFrame({
    'Feature': ['Open', 'High', 'Low'],
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(importance)

  Feature  Importance
2     Low    0.734776
1    High    0.262190
0    Open    0.003033


## ***8. Future Work***

In [15]:
# Save the best model
joblib.dump(gb_model, 'yes_bank_stock_predictor.joblib')
print("Model Saved!")

Model Saved!


In [16]:
# Load and Predict
loaded_model = joblib.load('yes_bank_stock_predictor.joblib')
sample = np.array([[25.0, 27.0, 24.0]])
print("Predicted Close Price:", loaded_model.predict(scaler.transform(sample))[0])

Predicted Close Price: 19.338966975082773


# **Future Enhancements:**
- Implement Time Series models (ARIMA, Prophet, LSTM).
- Include external factors like news sentiment, RBI policies, and macroeconomic indicators.
- Deploy the model as a web app using Streamlit or Flask.
- Real-time prediction using live stock data API.

# **Business Impact:**
This model can assist investors, traders, and financial analysts in making data-driven decisions and managing risk effectively.


# **Conclusion**


This project successfully performed comprehensive Exploratory Data Analysis and built multiple regression models to predict Yes Bank’s monthly Closing Price. After thorough evaluation, the **Tuned Gradient Boosting Regressor** emerged as the best model with excellent accuracy (high R² and low RMSE).

**Key Takeaways:**
- Open, High, and Low prices are highly predictive of the Closing price.
- The model effectively captured the sharp decline during the 2020 crisis.
- Outlier handling through capping preserved important market events.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***